<a href="https://colab.research.google.com/github/devangc006/Ai-ML/blob/main/ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# 1. Load the dataset (Breast Cancer dataset for high precision/accuracy classification)
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features (essential for neural networks)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training shape: {X_train_scaled.shape}")
print(f"Testing shape: {X_test_scaled.shape}")

Training shape: (455, 30)
Testing shape: (114, 30)


In [2]:
# 2. Define the ANN Model Architecture
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.3),  # Prevents overfitting to preserve generalization
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')  # Sigmoid output for binary classification
])

# Compile the model with metrics tracking Accuracy and Precision
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)

# Implement Early Stopping to halt training once validation loss stops improving
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train the model
history = model.fit(
    X_train_scaled, y_train,
    validation_data=(X_test_scaled, y_test),
    epochs=100,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - accuracy: 0.8110 - loss: 0.4614 - precision: 0.8419 - recall: 0.8596 - val_accuracy: 0.9386 - val_loss: 0.2830 - val_precision: 0.9333 - val_recall: 0.9722
Epoch 2/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9385 - loss: 0.2285 - precision: 0.9298 - recall: 0.9754 - val_accuracy: 0.9386 - val_loss: 0.1620 - val_precision: 0.9333 - val_recall: 0.9722
Epoch 3/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9538 - loss: 0.1664 - precision: 0.9459 - recall: 0.9825 - val_accuracy: 0.9386 - val_loss: 0.1192 - val_precision: 0.9333 - val_recall: 0.9722
Epoch 4/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9582 - loss: 0.1218 - precision: 0.9586 - recall: 0.9754 - val_accuracy: 0.9737 - val_loss: 0.1024 - val_precision: 0.9859 - val_recall: 0.9722
Epoch 5/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9802 - loss: 0.1017 - precision: 0.9825 - recall: 0.9860 - val_accuracy: 0.9737 - val_loss: 0.

In [4]:
# 3. Evaluate Model and Print Performance Metrics

# Evaluate the model on test data to get loss and accuracy
eval_results = model.evaluate(X_test_scaled, y_test, verbose=0)
test_loss = eval_results[0]
test_accuracy = eval_results[1]

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# Generate predictions
y_pred = (model.predict(X_test_scaled, verbose=0) > 0.5).astype("int32")

# Compute Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

# Generate Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=data.target_names))

Test Loss: 0.0786
Test Accuracy: 0.9737

Confusion Matrix:
[[41  1]
 [ 2 70]]

Classification Report:
              precision    recall  f1-score   support

   malignant       0.95      0.98      0.96        42
      benign       0.99      0.97      0.98        72

    accuracy                           0.97       114
   macro avg       0.97      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114

